COBA INPUT DATASET SECARA ONLINE

In [7]:
import pandas as pd

URL = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"

try:
    df = pd.read_csv(URL)
    print("Berhasil memuat dari Jalur A.")
except Exception as e:
    print(f"Jalur A gagal ({e}), coba Jalur B (ucimlrepo) di bawah.")

Berhasil memuat dari Jalur A.


COBA INPUT DATASET VIA REPO

In [6]:
# =========================================================
# ALTERNATIF: memuat via library resmi UCI
# =========================================================
# Kelebihan: ikut membawa metadata & deskripsi variabel resmi.
# Kekurangan: perlu install dulu.
# Kalau Jalur A sudah berhasil, kita boleh melewati bagian ini.

!pip install -q ucimlrepo

from ucimlrepo import fetch_ucirepo

repo = fetch_ucirepo(id=352) # 352 = ID resmi dataset "Online Retail" di UCI
# Dataset ini murni transaksional, tidak punya kolom "target" untuk
# diprediksi, jadi repo.data.targets bernilai None. Alih-alih memakai pola
# umum pd.concat([repo.data.features, repo.data.targets], axis=1),
# gunakan repo.data.original, yang merekonstruksi tabel lengkapnya:

df = repo.data.original

print(repo.metadata.abstract) # ringkasan resmi dataset
print(repo.variables) # keterangan tiap kolom

KeyboardInterrupt: 

AWAL DATA EXPLORATION

In [8]:
print(df.shape) # jumlah baris & kolom
print(df.info()) # ringkasan tipe data tiap kolom

(541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB
None


MENANGANI MISSING VALUE

In [9]:
print(df.isna().sum()) # jumlah nilai hilang tiap kolom

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [10]:
missing_customer_id = df[df['CustomerID'].isna()]
print(missing_customer_id['StockCode'].value_counts().head(5)) # kode produk yang sering muncul tanpa CustomerID

StockCode
DOT       694
85099B    497
21931     414
22411     388
20712     372
Name: count, dtype: int64


In [11]:
# Untuk analisis TOTAL PENDAPATAN: pertahankan semua baris,
# CustomerID yang kosong tidak memengaruhi validitas nilai transaksinya

pendapatan_semua_transaksi = (df["Quantity"] * df["UnitPrice"]).sum()

# Untuk analisis PER PELANGGAN: keluarkan baris tanpa CustomerID,
# karena baris itu tidak bisa ditautkan ke satu pelanggan tertentu

df_per_pelanggan = df.dropna(subset=["CustomerID"])

total_pendapatan_per_pelanggan = (df_per_pelanggan["Quantity"] * df_per_pelanggan["UnitPrice"]).sum() / df_per_pelanggan["CustomerID"].nunique()
print(pendapatan_semua_transaksi)
print(total_pendapatan_per_pelanggan)

9747747.934
1898.459701280878


MENANGANI DUPLIKASI DATA

In [12]:
jumlah_duplikasi = df.duplicated().sum()
print(f"Jumlah baris duplikasi: {jumlah_duplikasi}")

Jumlah baris duplikasi: 5268


In [13]:
contoh_duplikasi = df[df.duplicated(keep=False)].sort_values(by=["InvoiceNo", "StockCode"])
print(contoh_duplikasi.head(10)) # contoh 10 baris duplikasi pertama

df = df.drop_duplicates()
print(f'Baris setelah duplikasi dihapus: {len(df)}')

    InvoiceNo StockCode                       Description  Quantity  \
494    536409     21866       UNION JACK FLAG LUGGAGE TAG         1   
517    536409     21866       UNION JACK FLAG LUGGAGE TAG         1   
485    536409     22111      SCOTTIE DOG HOT WATER BOTTLE         1   
539    536409     22111      SCOTTIE DOG HOT WATER BOTTLE         1   
489    536409     22866     HAND WARMER SCOTTY DOG DESIGN         1   
527    536409     22866     HAND WARMER SCOTTY DOG DESIGN         1   
521    536409     22900   SET 2 TEA TOWELS I LOVE LONDON          1   
537    536409     22900   SET 2 TEA TOWELS I LOVE LONDON          1   
565    536412     21448         12 DAISY PEGS IN WOOD BOX         2   
578    536412     21448         12 DAISY PEGS IN WOOD BOX         1   

         InvoiceDate  UnitPrice  CustomerID         Country  
494  12/1/2010 11:45       1.25     17908.0  United Kingdom  
517  12/1/2010 11:45       1.25     17908.0  United Kingdom  
485  12/1/2010 11:45       4.95 

MENDETEKSI OUTLIER : IQR VS Z-SCORE

In [14]:
Q1 = df["Quantity"].quantile(0.25)
Q3 = df["Quantity"].quantile(0.75)
IQR = Q3 - Q1

batas_bawah, batas_atas = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outlier_iqr = df[(df["Quantity"] < batas_bawah) | (df["Quantity"] > batas_atas)]
print(f"IQR: batas [{batas_bawah:.1f}, {batas_atas:.1f}], outlier: {len(outlier_iqr)}({len(outlier_iqr)/len(df)*100:.2f}%)")
    
mean_q, std_q = df["Quantity"].mean(), df["Quantity"].std()
z = (df["Quantity"] - mean_q) / std_q
outlier_z = df[z.abs() > 3]
print(f"Z-score: mean={mean_q:.2f}, std={std_q:.2f}, outlier: {len(outlier_z)}({len(outlier_z)/len(df)*100:.2f}%)")

IQR: batas [-12.5, 23.5], outlier: 58501(10.90%)
Z-score: mean=9.62, std=219.13, outlier: 343(0.06%)


MENANGANI OUTLIER: CAP, TRANSFORMASI, DAN/ATAU HAPUS

In [15]:
import numpy as np

# Pilihan 1: CAP (winsorizing) - ganti nilai ekstrem dengan batas IQR, jumlah baris tidak berubah
df_capped = df.copy()
df_capped["Quantity"] = df_capped["Quantity"].clip(lower=batas_bawah, upper=batas_atas)

# Pilihan 2: HAPUS - buang baris outlier sepenuhnya
df_removed = df[(df["Quantity"] >= batas_bawah) & (df["Quantity"] <= batas_atas)]

# Pilihan 3: TRANSFORMASI - kompres rentang nilai ekstrem lewat log, tanpa membuang satu baris pun
df_log = df.copy()
df_log["Quantity_log"] = np.sign(df_log["Quantity"]) * np.log1p(df_log["Quantity"].abs())

STRATEGI IMPUTASI: KAPAN MEAN/MEDIAN, KAPAN MICE

In [22]:
from pyampute.exploration.mcar_statistical_tests import MCARTest

np.random.seed(42)  # Untuk reproduktifitas
n = 500
usia = np.random.normal(38, 10, n)
pendapatan = 8 + 0.3*usia + np.random.normal(0, 3, n)
belanja = 2 + 0.4*pendapatan + np.random.normal(0, 2, n)
df_demo = pd.DataFrame({"usia": usia, "Pendapatan_Juta": pendapatan, "Belanja_Juta": belanja})

print(df_demo.head())

# Skenario A: kekosongan MCAR murni (dipilih acak, tidak bergantung variabel apa pun)
df_mcar = df_demo.copy()
idx_mcar = np.random.choice(n, size=int(n * 0.2), replace=False)
df_mcar.loc[idx_mcar, "pendapatan_juta"] = np.nan

# Skenario B: kekosongan MAR (orang lebih tua lebih sering tidak mengisi pendapatan)
df_mar = df_demo.copy()
peluang_hilang = 1 / (1 + np.exp(-(df_demo["usia"] - 45) / 3))
mask_mar = np.random.random(n) < peluang_hilang
df_mar.loc[mask_mar, "pendapatan_juta"] = np.nan

uji = MCARTest(method="little")
print(f"Little's test (skenario MCAR): p-value = {uji.little_mcar_test(df_mcar):.4f}")
print(f"Little's test (skenario MAR): p-value = {uji.little_mcar_test(df_mar):.4f}")

        usia  Pendapatan_Juta  Belanja_Juta
0  42.967142        23.668675     14.266181
1  36.617357        24.713457     13.734650
2  44.476885        17.147363      8.978206
3  53.230299        25.657997     10.969325
4  35.658466        16.745612     10.094691
Little's test (skenario MCAR): p-value = nan
Little's test (skenario MAR): p-value = nan
